# 03 · Análise exploratória e explicativa

Toda estimativa sai **em par**: arquivo entregue (sem peso) e BRFSS completo
(ponderado). A diferença entre as colunas **é** o resultado.

Com n = 253.680 todo p-valor dá zero — ele não distingue nada nesta escala.
Reportamos **tamanho de efeito** e **intervalo de confiança**.

> Documentos: [`docs/06`](../docs/06-analise-exploratoria.md) e [`docs/07`](../docs/07-analise-explicativa.md)


In [1]:
import sys, json
from pathlib import Path

# a raiz e onde existe src/ — funciona rodando de notebooks/ ou da raiz do repo
RAIZ = Path.cwd()
if not (RAIZ / "src").exists():
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np, pandas as pd
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 50)

GOLD = RAIZ / "data" / "processed" / "gold"
def ler(nome, base=GOLD):
    return json.loads((base / nome).read_text(encoding="utf-8"))


## Associações brutas — o arquivo entregue atenua os efeitos


In [2]:
eda = ler("_eda_comparativa.json")
b = pd.DataFrame(eda["binarias"])
b[["variavel", "A_%_exposto", "B_%_exposto", "delta_exposicao",
   "A_OR", "B_OR", "delta_OR_%", "B_V", "efeito_B"]].head(10)


,variavel,A_%_exposto,B_%_exposto,delta_exposicao,A_OR,B_OR,delta_OR_%,B_V,efeito_B
0,exame_colesterol,96.27,77.95,18.32,6.429,7.109,-9.6,0.1484,pequeno
1,hipertensao,42.90,31.98,10.92,5.038,6.813,-26.1,0.2931,pequeno
2,doenca_cardiaca,9.42,6.41,3.01,3.623,5.073,-28.6,0.1939,pequeno
3,dificuldade_caminhar,16.82,13.72,3.10,3.771,4.785,-21.2,0.2330,pequeno
4,avc,4.06,3.03,1.03,3.065,3.944,-22.3,0.1135,pequeno
5,colesterol_alto,42.41,36.48,5.93,3.255,3.877,-16.0,0.2255,pequeno
6,acesso_saude,95.11,87.86,7.25,1.265,1.809,-30.1,0.0494,desprezivel
7,fumante,44.32,41.33,2.99,1.420,1.585,-10.4,0.0711,desprezivel
8,sem_consulta_por_custo,8.42,13.23,-4.81,1.349,1.147,17.6,0.0148,desprezivel
9,sexo,44.03,48.66,-4.63,1.199,1.096,9.4,0.0140,desprezivel


O sinal de `delta_OR_%` é **negativo em 11 das 14** variáveis: o arquivo entregue
**subestima** as associações, e a atenuação chega a 30%. Quem analisar só o
arquivo subestima o efeito da hipertensão em um quarto.

E o maior V de Cramér é **0,293** — ainda "pequeno" por Cohen. **Diabetes é
multifatorial e nenhuma variável isolada o explica.**


## Gradientes ordinais — idade e a mortalidade seletiva


In [3]:
ordinais = {o["variavel"]: o for o in eda["ordinais"]}
idade = ordinais["idade_faixa"]
A = idade["A_prev_por_nivel"] if isinstance(idade["A_prev_por_nivel"], dict) else eval(idade["A_prev_por_nivel"])
B = idade["B_prev_por_nivel"] if isinstance(idade["B_prev_por_nivel"], dict) else eval(idade["B_prev_por_nivel"])
g = pd.DataFrame({"arquivo": pd.Series(A).astype(float),
                  "populacional": pd.Series(B).astype(float)})
g.index = ["18-24","25-29","30-34","35-39","40-44","45-49","50-54",
           "55-59","60-64","65-69","70-74","75-79","80+"]
display(g)
print(f"razão extremos — arquivo {idade['A_razao']}×   populacional {idade['B_razao']}×")


,arquivo,populacional
18-24,1.37,0.82
25-29,1.84,1.43
30-34,2.82,2.10
35-39,4.53,3.69
40-44,6.50,6.79
45-49,8.79,9.13
50-54,11.74,11.74
55-59,13.83,15.53
60-64,17.25,19.72
65-69,20.37,22.73


razão extremos — arquivo 15.96×   populacional 30.02×


Dois achados nesta tabela:

1. **O arquivo comprime o gradiente etário quase pela metade** (15,96× contra 30,02×);
2. **não é monotônico** — sobe até 75–79 (24,66%) e **cai em 80+ (19,67%)**.
   É **mortalidade seletiva**: diabéticos têm menor chance de chegar aos 80.
   Modelar idade como linear ignora essa inflexão.


## Odds ratio ajustado — M1 (risco puro)


In [4]:
mod = ler("_modelo_explicativo.json")
m1 = pd.DataFrame(mod["comparacao_M1_entre_bases"]).T
m1.sort_values("B_OR", ascending=False)[["A_OR", "B_OR", "B_ic", "atenuacao_%"]]


,A_OR,B_OR,B_ic,atenuacao_%
hipertensao,2.452,2.394,[2.26; 2.54],2.4
colesterol_alto,1.897,2.0,[1.89; 2.11],-5.2
idade_faixa,1.48,1.645,[1.60; 1.70],-10.0
imc,1.575,1.589,[1.55; 1.63],-0.9
doenca_cardiaca,1.562,1.556,[1.45; 1.68],0.3
sexo,1.288,1.24,[1.18; 1.31],3.8
avc,1.348,1.215,[1.09; 1.35],10.9
fumante,1.044,1.082,[1.03; 1.14],-3.5
saude_mental_dias,1.076,1.058,[1.03; 1.08],1.7
frutas,0.942,0.943,[0.89; 1.00],-0.1


**Correção importante:** na análise *bruta* o arquivo atenuava os OR em até 30%.
No modelo **ajustado**, a maior divergência é 11%. O ajuste multivariado absorve
a maior parte do viés de seleção — porque a seleção operou por idade, renda e
acesso, que agora são covariáveis.

Isso **reabilita parcialmente** o arquivo entregue para análise multivariada.
Registrar isso corta contra a narrativa mais fácil, e é o que separa análise de retórica.


## A mediação que muda a interpretação


In [5]:
est = pd.DataFrame(mod["estabilidade_M1_M2_M3"]).T
est.loc[["atividade_fisica", "renda_faixa", "saude_geral", "hipertensao",
         "doenca_cardiaca", "saude_mental_dias"]]


,M1_risco_puro,M2_clinico,M3_completo,desloc_M1_M3_%
atividade_fisica,0.852,0.988,0.978,14.8
renda_faixa,0.818,0.907,0.904,10.5
saude_geral,NaN,1.781,1.787,NaN
hipertensao,2.394,2.110,2.080,-13.1
doenca_cardiaca,1.556,1.261,1.252,-19.5
saude_mental_dias,1.058,0.951,0.951,-10.1


```
M1 (risco puro)        atividade_fisica  OR 0,852   protetor
M2 (+ saúde geral…)    atividade_fisica  OR 0,988   some
```

Ao entrar `saude_geral`, o efeito da atividade física **evapora**. Isso é
**mediação** — e é a leitura errada mais provável do trabalho inteiro.
**M2 e M3 não podem ser lidos como "atividade física não importa".**


## O alvo não é ordinal — e o teste ingênuo não detecta


In [6]:
pvd = pd.DataFrame(mod["pre_vs_diabetes"]).T
pvd[~pvd["ic_sobrepoe"].astype(bool)][
    ["or_pre_vs_sem", "ic_pre", "or_diab_vs_sem", "ic_diab", "razao_diab_pre"]]


,or_pre_vs_sem,ic_pre,or_diab_vs_sem,ic_diab,razao_diab_pre
hipertensao,1.47,[1.29; 1.68],2.106,[1.98; 2.24],1.43
doenca_cardiaca,0.923,[0.76; 1.12],1.248,[1.15; 1.35],1.35
saude_geral,1.339,[1.24; 1.44],1.802,[1.74; 1.87],1.35
sexo,1.003,[0.89; 1.14],1.256,[1.19; 1.33],1.25
imc,1.304,[1.24; 1.37],1.526,[1.49; 1.57],1.17
escolaridade,0.845,[0.80; 0.90],0.965,[0.94; 0.99],1.14
idade_faixa,1.439,[1.34; 1.54],1.642,[1.59; 1.70],1.14
saude_mental_dias,1.072,[1.02; 1.13],0.954,[0.93; 0.98],0.89
alcool_excessivo,0.872,[0.66; 1.15],0.519,[0.45; 0.60],0.6


**Nove variáveis com IC disjunto**, e duas **invertem de direção**.
`sexo` tem OR 1,00 no pré-diabetes e 1,26 no diabetes.

Lição metodológica: o teste por *logits cumulativos* **não rejeitou** — divergência
máxima de 8,6%. É falso negativo: com a classe 1 valendo 1,6%, os dois contrastes
são quase idênticos por construção. **Teste de Brant por cortes cumulativos é
inadequado quando uma classe é rara.**
